# Sumo Graph Analytics — Snowflake Pipeline

This notebook implements the full graph analytics pipeline for professional sumo bout analysis using Neo4j Graph Analytics for Snowflake.

**Pipeline Overview:**
1. Setup and permissions
2. Data preparation and filtering
3. Rank weight definition
4. Edge and node table construction
5. PageRank via Neo4j GDS
6. DOMINATES graph construction
7. Betweenness Centrality via Neo4j GDS
8. Rock-paper-scissors cycle enumeration
9. Chaos Score composite metric

**Prerequisites:**
- Snowflake account with ACCOUNTADMIN access
- Neo4j Graph Analytics native app installed as `NEO4J_GRAPH_ANALYTICS`
- Source table: `DEMO_DB.PUBLIC.DEVREL_SUMO_DB_20260512`

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark import Session

# Update connection parameters to match your environment
connection_params = {
    'account': '<your_account>',
    'user': '<your_user>',
    'password': '<your_password>',
    'role': 'ACCOUNTADMIN',
    'warehouse': 'GDSONSNOWFLAKE',
    'database': 'DEMO_DB',
    'schema': 'PUBLIC'
}

session = Session.builder.configs(connection_params).create()
print('Connected to Snowflake')

## Step 1: Setup and Permissions

Grant the Neo4j Graph Analytics application access to the database and warehouse. Run once as ACCOUNTADMIN.

In [ ]:
setup_sql = """
-- Create roles for GDS users and admins
CREATE ROLE IF NOT EXISTS gds_user_role;
CREATE ROLE IF NOT EXISTS gds_admin_role;
GRANT APPLICATION ROLE NEO4J_GRAPH_ANALYTICS.app_user TO ROLE gds_user_role;
GRANT APPLICATION ROLE NEO4J_GRAPH_ANALYTICS.app_admin TO ROLE gds_admin_role;

-- Create database role and grant to user/admin roles and GDS application
CREATE DATABASE ROLE IF NOT EXISTS gds_db_role;
GRANT DATABASE ROLE gds_db_role TO ROLE gds_user_role;
GRANT DATABASE ROLE gds_db_role TO APPLICATION NEO4J_GRAPH_ANALYTICS;

-- Grant access to DEMO_DB
GRANT USAGE ON DATABASE DEMO_DB TO ROLE gds_user_role;
GRANT USAGE ON SCHEMA DEMO_DB.PUBLIC TO ROLE gds_user_role;
GRANT SELECT ON ALL TABLES IN DATABASE DEMO_DB TO DATABASE ROLE gds_db_role;
GRANT ALL PRIVILEGES ON FUTURE TABLES IN SCHEMA DEMO_DB.PUBLIC TO DATABASE ROLE gds_db_role;
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA DEMO_DB.PUBLIC TO DATABASE ROLE gds_db_role;
GRANT CREATE TABLE ON SCHEMA DEMO_DB.PUBLIC TO DATABASE ROLE gds_db_role;
GRANT CREATE VIEW ON SCHEMA DEMO_DB.PUBLIC TO DATABASE ROLE gds_db_role;
GRANT ALL PRIVILEGES ON FUTURE VIEWS IN SCHEMA DEMO_DB.PUBLIC TO DATABASE ROLE gds_db_role;
GRANT ALL PRIVILEGES ON ALL VIEWS IN SCHEMA DEMO_DB.PUBLIC TO DATABASE ROLE gds_db_role;

-- Compute and warehouse access
GRANT USAGE ON WAREHOUSE GDSONSNOWFLAKE TO APPLICATION NEO4J_GRAPH_ANALYTICS;
"""

for statement in setup_sql.strip().split(';'):
    stmt = statement.strip()
    if stmt:
        session.sql(stmt).collect()

print('Setup complete')

## Step 2: Data Preparation

Filter to Makuuchi division bouts between January 2021 and November 2025.
This removes lower division noise and the historical data skew caused by incomplete bout records in earlier basho.

In [ ]:
session.sql("""
CREATE OR REPLACE VIEW DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT AS
SELECT *
FROM DEMO_DB.PUBLIC.DEVREL_SUMO_DB_20260512
WHERE BASHO_ID BETWEEN 202101 AND 202511
  AND (
      LEFT_RANK_CLASS  IN ('Y','O','S','K','M')
   OR RIGHT_RANK_CLASS IN ('Y','O','S','K','M')
  )
""").collect()

count = session.sql("SELECT COUNT(DISTINCT BOUT_ID) AS bouts FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT").collect()
print(f'Bouts in analysis window: {count[0]["BOUTS"]}')

In [ ]:
# Build the active rikishi roster
# Groups by RIKISHI_ID only to handle wrestlers who changed ring names (shikona)
# during the analysis window. Most recent name is selected via BASHO_ID DESC.
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE AS
WITH WINS AS (
    SELECT
        WINNER_ID               AS RIKISHI_ID,
        COUNT(DISTINCT BOUT_ID) AS WIN_COUNT
    FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT
    GROUP BY WINNER_ID
),
LOSSES AS (
    SELECT
        LOSER_ID                AS RIKISHI_ID,
        COUNT(DISTINCT BOUT_ID) AS LOSS_COUNT
    FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT
    GROUP BY LOSER_ID
),
TOTALS AS (
    SELECT
        COALESCE(w.RIKISHI_ID, l.RIKISHI_ID) AS RIKISHI_ID,
        COALESCE(w.WIN_COUNT, 0)              AS WIN_COUNT,
        COALESCE(l.LOSS_COUNT, 0)             AS LOSS_COUNT,
        COALESCE(w.WIN_COUNT, 0)
            + COALESCE(l.LOSS_COUNT, 0)       AS TOTAL_BOUTS
    FROM WINS w
    FULL OUTER JOIN LOSSES l ON l.RIKISHI_ID = w.RIKISHI_ID
    HAVING TOTAL_BOUTS >= 20
),
LATEST_NAME AS (
    SELECT
        WINNER_ID   AS RIKISHI_ID,
        WINNER_NAME AS RIKISHI_NAME
    FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY WINNER_ID
        ORDER BY BASHO_ID DESC
    ) = 1
)
SELECT
    t.RIKISHI_ID,
    n.RIKISHI_NAME,
    t.WIN_COUNT,
    t.LOSS_COUNT,
    t.TOTAL_BOUTS
FROM TOTALS t
JOIN LATEST_NAME n ON n.RIKISHI_ID = t.RIKISHI_ID
""").collect()

roster = session.sql("SELECT COUNT(*) AS rikishi FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE").collect()
print(f'Active rikishi (20+ bouts): {roster[0]["RIKISHI"]}')

## Step 3: Rank Weight Table

Define prestige weights for each rank class. A win over a Yokozuna carries significantly more weight than a win over a lower Maegashira. These weights feed into the PageRank edge weighting.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_RANK_WEIGHTS (
    RANK_CLASS      VARCHAR,
    RANK_NUMBER_MAX NUMBER,
    WEIGHT          FLOAT,
    DIVISION        VARCHAR,
    NOTES           VARCHAR
)
""").collect()

session.sql("""
INSERT INTO DEMO_DB.PUBLIC.SUMO_RANK_WEIGHTS VALUES
    ('Y',  99, 8.0,  'Makuuchi', 'Yokozuna - cannot be demoted, highest prestige'),
    ('O',  99, 6.0,  'Makuuchi', 'Ozeki - second highest'),
    ('S',  99, 4.0,  'Makuuchi', 'Sekiwake - third highest sanyaku'),
    ('K',  99, 3.0,  'Makuuchi', 'Komusubi - fourth highest sanyaku'),
    ('M',   3, 2.0,  'Makuuchi', 'Maegashira 1-3 - upper joi, face sanyaku regularly'),
    ('M',   8, 1.5,  'Makuuchi', 'Maegashira 4-8 - mid joi'),
    ('M',  99, 1.0,  'Makuuchi', 'Maegashira 9+ - lower maegashira'),
    ('J',  99, 0.6,  'Juryo',    'Juryo - paid professional, one below Makuuchi'),
    ('Ms', 99, 0.3,  'Makushita','Makushita - final barrier before sekitori status'),
    ('Sd', 99, 0.15, 'Sandanme', 'Sandanme - development division'),
    ('Jd', 99, 0.08, 'Jonidan',  'Jonidan - lower development division'),
    ('Jk', 99, 0.04, 'Jonokuchi','Jonokuchi - entry level division'),
    ('Mz', 99, 0.02, 'Maezumo',  'Maezumo - exhibition bouts before official ranking')
""").collect()

print('Rank weights created')
session.sql("SELECT * FROM DEMO_DB.PUBLIC.SUMO_RANK_WEIGHTS ORDER BY WEIGHT DESC").to_pandas()

## Step 4: Build Edge and Node Tables for PageRank

The edge table flows from winner (sourceNodeId) to loser (targetNodeId), weighted by the loser's rank prestige. These aliases are required by the Neo4j GDS PageRank procedure.

We use REVERSE orientation when calling PageRank because our edges point winner to loser. Without reversal, prestige would flow toward losers. Reversing ensures prestige accumulates on winners of high-value bouts.

In [ ]:
# Build rank-weighted edge table
# One row per bout, weighted by the loser's rank at time of bout
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_DEFEATED_EDGES AS
SELECT
    b.WINNER_ID                             AS sourceNodeId,
    b.LOSER_ID                              AS targetNodeId,
    CASE
        WHEN b.WINNER_ID = b.LEFT_ID  THEN b.RIGHT_RANK_CLASS
        WHEN b.WINNER_ID = b.RIGHT_ID THEN b.LEFT_RANK_CLASS
    END                                     AS LOSER_RANK_CLASS,
    CASE
        WHEN b.WINNER_ID = b.LEFT_ID  THEN b.RIGHT_RANK_NUMBER
        WHEN b.WINNER_ID = b.RIGHT_ID THEN b.LEFT_RANK_NUMBER
    END                                     AS LOSER_RANK_NUMBER,
    COALESCE(w.WEIGHT, 1.0)                 AS BOUT_WEIGHT
FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT b
LEFT JOIN DEMO_DB.PUBLIC.SUMO_RANK_WEIGHTS w
    ON w.RANK_CLASS = CASE
        WHEN b.WINNER_ID = b.LEFT_ID  THEN b.RIGHT_RANK_CLASS
        WHEN b.WINNER_ID = b.RIGHT_ID THEN b.LEFT_RANK_CLASS
       END
   AND w.RANK_NUMBER_MAX >= CASE
        WHEN b.WINNER_ID = b.LEFT_ID  THEN b.RIGHT_RANK_NUMBER
        WHEN b.WINNER_ID = b.RIGHT_ID THEN b.LEFT_RANK_NUMBER
       END
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY b.BOUT_ID
    ORDER BY w.RANK_NUMBER_MAX ASC
) = 1
""").collect()

# Aggregate to one weighted edge per winner/loser pair
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_DEFEATED_EDGES_AGG AS
SELECT
    sourceNodeId,
    targetNodeId,
    SUM(BOUT_WEIGHT) AS WEIGHT,
    COUNT(*)         AS BOUT_COUNT
FROM DEMO_DB.PUBLIC.SUMO_DEFEATED_EDGES
WHERE sourceNodeId IN (SELECT RIKISHI_ID FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE)
  AND targetNodeId IN (SELECT RIKISHI_ID FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE)
GROUP BY sourceNodeId, targetNodeId
""").collect()

# Node table - nodeId alias required by GDS
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_GDS_NODES AS
SELECT
    RIKISHI_ID   AS nodeId,
    RIKISHI_NAME AS NAME
FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE
""").collect()

edges = session.sql("SELECT COUNT(*) AS cnt FROM DEMO_DB.PUBLIC.SUMO_DEFEATED_EDGES_AGG").collect()
nodes = session.sql("SELECT COUNT(*) AS cnt FROM DEMO_DB.PUBLIC.SUMO_GDS_NODES").collect()
print(f'Nodes: {nodes[0]["CNT"]} | Edges: {edges[0]["CNT"]}')

## Step 5: PageRank via Neo4j Graph Analytics

PageRank in our defeat network measures the quality of a wrestler's victories, not just the count. Beating a wrestler who beat many strong opponents propagates more prestige than accumulating wins against weaker competition.

We use REVERSE orientation so prestige flows toward winners rather than losers.

In [ ]:
# Run rank-weighted PageRank
# REVERSE orientation: flips edge direction so prestige flows to winners
# dampingFactor 0.85: 85% probability prestige continues through chain
# maxIterations 20: cap on redistribution passes (increase if didConverge=false)
session.sql("""
CALL NEO4J_GRAPH_ANALYTICS.graph.pagerank('CPU_X64_L', {
  'project': {
    'defaultTablePrefix': 'DEMO_DB.PUBLIC',
    'nodeTables': ['SUMO_GDS_NODES'],
    'relationshipTables': {
      'SUMO_DEFEATED_EDGES_AGG': {
        'sourceTable': 'SUMO_GDS_NODES',
        'targetTable': 'SUMO_GDS_NODES',
        'orientation': 'REVERSE',
        'weightProperty': 'WEIGHT'
      }
    }
  },
  'compute': {
    'dampingFactor': 0.85,
    'maxIterations': 20,
    'relationshipWeightProperty': 'WEIGHT'
  },
  'write': [
    {
      'nodeLabel': 'SUMO_GDS_NODES',
      'outputTable': 'DEMO_DB.PUBLIC.SUMO_PAGERANK_RESULTS'
    }
  ]
})
""").collect()

# Join results back to rikishi names
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_PAGERANK AS
SELECT
    n.RIKISHI_ID,
    n.RIKISHI_NAME,
    p.PAGERANK,
    n.TOTAL_BOUTS
FROM DEMO_DB.PUBLIC.SUMO_PAGERANK_RESULTS p
JOIN DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE n ON n.RIKISHI_ID = p.NODEID
ORDER BY PAGERANK DESC
""").collect()

print('PageRank complete. Top 10:')
session.sql("""
SELECT RIKISHI_NAME, ROUND(PAGERANK, 4) AS PAGERANK_SCORE
FROM DEMO_DB.PUBLIC.SUMO_PAGERANK
ORDER BY PAGERANK DESC
LIMIT 10
""").to_pandas()

In [ ]:
# Compare raw wins vs PageRank
# Positive delta = beats quality opponents above what win count suggests
# Negative delta = win count inflated by weaker opposition
session.sql("""
SELECT
    r.RIKISHI_ID,
    r.RIKISHI_NAME,
    COUNT(DISTINCT w.BOUT_ID)                                               AS RAW_WIN_COUNT,
    r.TOTAL_BOUTS,
    ROUND(COUNT(DISTINCT w.BOUT_ID) / r.TOTAL_BOUTS::FLOAT, 3)             AS WIN_RATE,
    ROUND(p.PAGERANK, 4)                                                    AS PAGERANK_SCORE,
    ROUND(COUNT(DISTINCT w.BOUT_ID)::FLOAT / MAX(COUNT(DISTINCT w.BOUT_ID)) OVER(), 3) AS NORM_WIN_COUNT,
    ROUND((p.PAGERANK - MIN(p.PAGERANK) OVER()) /
          (MAX(p.PAGERANK) OVER() - MIN(p.PAGERANK) OVER()), 3)             AS NORM_PAGERANK,
    ROUND(
        ((p.PAGERANK - MIN(p.PAGERANK) OVER()) /
         (MAX(p.PAGERANK) OVER() - MIN(p.PAGERANK) OVER()))
        -
        (COUNT(DISTINCT w.BOUT_ID)::FLOAT / MAX(COUNT(DISTINCT w.BOUT_ID)) OVER())
    , 3)                                                                    AS PAGERANK_VS_WINS_DELTA
FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT w
JOIN DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE r ON r.RIKISHI_ID = w.WINNER_ID
JOIN DEMO_DB.PUBLIC.SUMO_PAGERANK p       ON p.RIKISHI_ID = w.WINNER_ID
GROUP BY r.RIKISHI_ID, r.RIKISHI_NAME, r.TOTAL_BOUTS, p.PAGERANK
ORDER BY PAGERANK_SCORE DESC
LIMIT 20
""").to_pandas()

## Step 6: Build the DOMINATES Graph

Thousands of individual bouts create a dense and noisy graph where volume obscures the underlying competitive structure. Collapsing each head-to-head matchup into a single directed edge pointing toward the net winner cuts through that noise and surfaces the true dominance relationships that define the division.

Exact ties (equal wins on both sides) are excluded as there is no dominant direction.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_DOMINATES AS
WITH HEAD_TO_HEAD AS (
    SELECT
        LEAST(WINNER_ID, LOSER_ID)                             AS RIKISHI_A,
        GREATEST(WINNER_ID, LOSER_ID)                          AS RIKISHI_B,
        SUM(CASE WHEN WINNER_ID < LOSER_ID THEN 1 ELSE 0 END)  AS A_WINS,
        SUM(CASE WHEN WINNER_ID > LOSER_ID THEN 1 ELSE 0 END)  AS B_WINS,
        COUNT(DISTINCT BOUT_ID)                                AS TOTAL_BOUTS
    FROM DEMO_DB.PUBLIC.SUMO_BOUTS_RECENT
    WHERE WINNER_ID IN (SELECT RIKISHI_ID FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE)
      AND LOSER_ID  IN (SELECT RIKISHI_ID FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE)
    GROUP BY
        LEAST(WINNER_ID, LOSER_ID),
        GREATEST(WINNER_ID, LOSER_ID)
    HAVING A_WINS <> B_WINS
)
SELECT
    CASE WHEN A_WINS > B_WINS THEN RIKISHI_A ELSE RIKISHI_B END AS SOURCE_NODE_ID,
    CASE WHEN A_WINS > B_WINS THEN RIKISHI_B ELSE RIKISHI_A END AS TARGET_NODE_ID,
    ABS(A_WINS - B_WINS)                                         AS MARGIN,
    TOTAL_BOUTS
FROM HEAD_TO_HEAD
""").collect()

dominates = session.sql("SELECT COUNT(*) AS cnt FROM DEMO_DB.PUBLIC.SUMO_DOMINATES").collect()
print(f'DOMINATES relationships created: {dominates[0]["CNT"]}')

## Step 7: Betweenness Centrality via Neo4j Graph Analytics

Where PageRank asked who beat the strongest opponents, Betweenness asks who sits on the most paths through the dominance hierarchy. A high Betweenness score identifies the wrestlers whose results connect the top of the division to the bottom.

We run Betweenness on the DOMINATES graph with NATURAL (directed) orientation because direction matters here. Betweenness on an undirected graph would treat wins and losses as equivalent connections, losing the hierarchical information entirely.

In [ ]:
# Build node table for DOMINATES graph
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_DOMINATES_NODES AS
SELECT DISTINCT SOURCE_NODE_ID AS NODEID FROM DEMO_DB.PUBLIC.SUMO_DOMINATES
UNION
SELECT DISTINCT TARGET_NODE_ID FROM DEMO_DB.PUBLIC.SUMO_DOMINATES
""").collect()

# Run Betweenness Centrality on DOMINATES graph
session.sql("""
CALL NEO4J_GRAPH_ANALYTICS.graph.betweenness('CPU_X64_L', {
  'project': {
    'defaultTablePrefix': 'DEMO_DB.PUBLIC',
    'nodeTables': ['SUMO_DOMINATES_NODES'],
    'relationshipTables': {
      'SUMO_DOMINATES': {
        'sourceTable': 'SUMO_DOMINATES_NODES',
        'targetTable': 'SUMO_DOMINATES_NODES',
        'orientation': 'NATURAL'
      }
    }
  },
  'compute': {},
  'write': [
    {
      'nodeLabel': 'SUMO_DOMINATES_NODES',
      'outputTable': 'DEMO_DB.PUBLIC.SUMO_BETWEENNESS_RESULTS'
    }
  ]
})
""").collect()

# Join results back to rikishi names
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_BETWEENNESS AS
SELECT
    r.RIKISHI_ID,
    r.RIKISHI_NAME,
    ROUND(b.BETWEENNESS, 4) AS BETWEENNESS_SCORE
FROM DEMO_DB.PUBLIC.SUMO_BETWEENNESS_RESULTS b
JOIN DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE r ON r.RIKISHI_ID = b.NODEID
ORDER BY BETWEENNESS_SCORE DESC
""").collect()

print('Betweenness Centrality complete. Top 10:')
session.sql("""
SELECT RIKISHI_NAME, BETWEENNESS_SCORE
FROM DEMO_DB.PUBLIC.SUMO_BETWEENNESS
ORDER BY BETWEENNESS_SCORE DESC
LIMIT 10
""").to_pandas()

In [ ]:
# Combined PageRank and Betweenness view
# High PageRank + High Betweenness = elite structural pillar
# High PageRank + Low Betweenness  = dominant but isolated
# Low PageRank  + High Betweenness = structural bridge without elite prestige
session.sql("""
SELECT
    r.RIKISHI_ID,
    r.RIKISHI_NAME,
    r.WIN_COUNT                                                         AS RAW_WINS,
    r.TOTAL_BOUTS,
    ROUND(r.WIN_COUNT::FLOAT / r.TOTAL_BOUTS, 3)                       AS WIN_RATE,
    ROUND(p.PAGERANK, 4)                                                AS PAGERANK_SCORE,
    ROUND(b.BETWEENNESS_SCORE, 4)                                       AS BETWEENNESS_SCORE,
    ROUND(p.PAGERANK / MAX(p.PAGERANK) OVER(), 3)                      AS NORM_PAGERANK,
    ROUND((b.BETWEENNESS_SCORE - MIN(b.BETWEENNESS_SCORE) OVER()) /
          NULLIF(MAX(b.BETWEENNESS_SCORE) OVER()
               - MIN(b.BETWEENNESS_SCORE) OVER(), 0), 3)               AS NORM_BETWEENNESS
FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE r
JOIN DEMO_DB.PUBLIC.SUMO_PAGERANK       p ON p.RIKISHI_ID = r.RIKISHI_ID
JOIN DEMO_DB.PUBLIC.SUMO_BETWEENNESS    b ON b.RIKISHI_ID = r.RIKISHI_ID
ORDER BY PAGERANK_SCORE DESC
LIMIT 20
""").to_pandas()

## Step 8: Rock-Paper-Scissors Cycle Enumeration

Not every rivalry in sumo follows a clean hierarchy. Sometimes wrestler A consistently beats B, B consistently beats C, and C consistently beats A. These non-transitive cycles reveal the limits of any linear ranking system and show that the current Makuuchi division cannot be reduced to a simple leaderboard.

In [ ]:
# Enumerate all 3-node rock-paper-scissors cycles in the DOMINATES graph
# Deduplication via SOURCE_NODE_ID ordering prevents counting rotations of the same cycle
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_RPS_CYCLES AS
SELECT DISTINCT
    a.SOURCE_NODE_ID                                            AS RIKISHI_A_ID,
    na.RIKISHI_NAME                                             AS RIKISHI_A,
    b.SOURCE_NODE_ID                                            AS RIKISHI_B_ID,
    nb.RIKISHI_NAME                                             AS RIKISHI_B,
    c.SOURCE_NODE_ID                                            AS RIKISHI_C_ID,
    nc.RIKISHI_NAME                                             AS RIKISHI_C,
    a.MARGIN                                                    AS A_BEATS_B_MARGIN,
    b.MARGIN                                                    AS B_BEATS_C_MARGIN,
    c.MARGIN                                                    AS C_BEATS_A_MARGIN,
    a.MARGIN + b.MARGIN + c.MARGIN                              AS TOTAL_CYCLE_MARGIN
FROM DEMO_DB.PUBLIC.SUMO_DOMINATES a
JOIN DEMO_DB.PUBLIC.SUMO_DOMINATES b
    ON  b.SOURCE_NODE_ID = a.TARGET_NODE_ID
JOIN DEMO_DB.PUBLIC.SUMO_DOMINATES c
    ON  c.SOURCE_NODE_ID = b.TARGET_NODE_ID
    AND c.TARGET_NODE_ID = a.SOURCE_NODE_ID
JOIN DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE na ON na.RIKISHI_ID = a.SOURCE_NODE_ID
JOIN DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE nb ON nb.RIKISHI_ID = b.SOURCE_NODE_ID
JOIN DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE nc ON nc.RIKISHI_ID = c.SOURCE_NODE_ID
WHERE a.SOURCE_NODE_ID < b.SOURCE_NODE_ID
  AND a.SOURCE_NODE_ID < c.SOURCE_NODE_ID
""").collect()

cycles = session.sql("SELECT COUNT(*) AS cnt FROM DEMO_DB.PUBLIC.SUMO_RPS_CYCLES").collect()
print(f'Rock-paper-scissors cycles found: {cycles[0]["CNT"]}')

print('\nTop 10 cycles by total margin:')
session.sql("""
SELECT RIKISHI_A, A_BEATS_B_MARGIN, RIKISHI_B,
       B_BEATS_C_MARGIN, RIKISHI_C, C_BEATS_A_MARGIN,
       TOTAL_CYCLE_MARGIN
FROM DEMO_DB.PUBLIC.SUMO_RPS_CYCLES
ORDER BY TOTAL_CYCLE_MARGIN DESC
LIMIT 10
""").to_pandas()

In [ ]:
# Count how many cycles each rikishi appears in
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_CYCLE_APPEARANCES AS
SELECT
    RIKISHI_ID,
    RIKISHI_NAME,
    COUNT(*) AS CYCLE_COUNT
FROM (
    SELECT RIKISHI_A_ID AS RIKISHI_ID, RIKISHI_A AS RIKISHI_NAME FROM DEMO_DB.PUBLIC.SUMO_RPS_CYCLES
    UNION ALL
    SELECT RIKISHI_B_ID, RIKISHI_B FROM DEMO_DB.PUBLIC.SUMO_RPS_CYCLES
    UNION ALL
    SELECT RIKISHI_C_ID, RIKISHI_C FROM DEMO_DB.PUBLIC.SUMO_RPS_CYCLES
)
GROUP BY RIKISHI_ID, RIKISHI_NAME
ORDER BY CYCLE_COUNT DESC
""").collect()

session.sql("""
SELECT RIKISHI_NAME, CYCLE_COUNT
FROM DEMO_DB.PUBLIC.SUMO_CYCLE_APPEARANCES
ORDER BY CYCLE_COUNT DESC
LIMIT 15
""").to_pandas()

## Step 9: Chaos Score

The Chaos Score is a composite metric that chains all three analyses together. A high Chaos Score identifies wrestlers who are simultaneously elite (PageRank), structurally important (Betweenness), and resistant to simple linear ranking (cycle involvement).

Each metric is normalized to [0,1] and summed with equal weighting. A wrestler who scores well across all three dimensions ranks higher than a specialist who dominates only one.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE DEMO_DB.PUBLIC.SUMO_CHAOS_SCORE AS
SELECT
    r.RIKISHI_ID,
    r.RIKISHI_NAME,
    r.WIN_COUNT                                                 AS RAW_WINS,
    r.TOTAL_BOUTS,
    ROUND(r.WIN_COUNT::FLOAT / r.TOTAL_BOUTS, 3)               AS WIN_RATE,
    ROUND(p.PAGERANK, 4)                                        AS PAGERANK_SCORE,
    ROUND(b.BETWEENNESS_SCORE, 4)                               AS BETWEENNESS_SCORE,
    COALESCE(ca.CYCLE_COUNT, 0)                                 AS CYCLE_COUNT,
    -- Normalize each metric to [0,1]
    ROUND(p.PAGERANK /
          MAX(p.PAGERANK) OVER(), 3)                            AS NORM_PAGERANK,
    ROUND((b.BETWEENNESS_SCORE - MIN(b.BETWEENNESS_SCORE) OVER()) /
          NULLIF(MAX(b.BETWEENNESS_SCORE) OVER()
               - MIN(b.BETWEENNESS_SCORE) OVER(), 0), 3)        AS NORM_BETWEENNESS,
    ROUND(COALESCE(ca.CYCLE_COUNT, 0)::FLOAT /
          NULLIF(MAX(ca.CYCLE_COUNT) OVER(), 0), 3)             AS NORM_CYCLES,
    -- Chaos Score: equal weighted sum of all three normalized metrics
    ROUND(
        (p.PAGERANK / MAX(p.PAGERANK) OVER())
        +
        ((b.BETWEENNESS_SCORE - MIN(b.BETWEENNESS_SCORE) OVER()) /
          NULLIF(MAX(b.BETWEENNESS_SCORE) OVER()
               - MIN(b.BETWEENNESS_SCORE) OVER(), 0))
        +
        (COALESCE(ca.CYCLE_COUNT, 0)::FLOAT /
          NULLIF(MAX(ca.CYCLE_COUNT) OVER(), 0))
    , 3)                                                        AS CHAOS_SCORE
FROM DEMO_DB.PUBLIC.SUMO_RIKISHI_ACTIVE        r
JOIN DEMO_DB.PUBLIC.SUMO_PAGERANK              p  ON p.RIKISHI_ID  = r.RIKISHI_ID
JOIN DEMO_DB.PUBLIC.SUMO_BETWEENNESS           b  ON b.RIKISHI_ID  = r.RIKISHI_ID
LEFT JOIN DEMO_DB.PUBLIC.SUMO_CYCLE_APPEARANCES ca ON ca.RIKISHI_ID = r.RIKISHI_ID
ORDER BY CHAOS_SCORE DESC
""").collect()

print('Chaos Score complete. Top 20:')
session.sql("""
SELECT
    RIKISHI_NAME,
    RAW_WINS,
    WIN_RATE,
    PAGERANK_SCORE,
    BETWEENNESS_SCORE,
    CYCLE_COUNT,
    CHAOS_SCORE
FROM DEMO_DB.PUBLIC.SUMO_CHAOS_SCORE
ORDER BY CHAOS_SCORE DESC
LIMIT 20
""").to_pandas()

## Summary

The pipeline produces three complementary views of the 2021-2025 Makuuchi competitive landscape:

| Metric | Algorithm | Question Answered |
|---|---|---|
| PageRank Score | Neo4j GDS PageRank | Who beat the strongest opponents? |
| Betweenness Score | Neo4j GDS Betweenness | Who holds the dominance hierarchy together? |
| Cycle Count | SQL self-join on DOMINATES | Who is caught in non-transitive rivalries? |
| Chaos Score | SQL composite | Who is elite, structural, and unpredictable simultaneously? |

Finding the best wrestlers in the current era of sumo turns out to be a far more complex question than it first appears. Raw win counts favor volume over quality. PageRank surfaces prestige but misses structure. Betweenness Centrality finds the structural pillars but ignores who they beat. Only by chaining these algorithms together does a complete picture emerge.